# Iris Pipelines: LDA+GBM, PCA+GBM, GBM baseline, and LdaBoost

This notebook provides a clean and reproducible comparison of several pipelines on the classic Iris dataset. It mirrors the original analysis while improving organization and clarity for supplementary material.

Sections:
- Dataset and preprocessing
- Reproducibility setup and constants
- LDA+GBM vs PCA+GBM (cross-validated)
- Baseline Gradient Boosting (cross-validated)
- LdaBoost (cross-validated)



In [ ]:
"""
Iris pipelines comparison notebook (cleaned and reproducible).

- Dataset: Iris (UCI)
- Preprocessing: none, except label encoding
- Models/Pipelines:
  1) LDA + Gradient Boosting
  2) PCA + Gradient Boosting
  3) Gradient Boosting baseline
  4) LdaBoost (custom boosting with LDA projections)
- Evaluation: Stratified K-Fold cross-validation accuracy

This notebook is intended as supplementary material.
"""
# Reproducibility
import os
import random
import numpy as np

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
os.environ["PYTHONHASHSEED"] = str(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# Tuning constants
N_ESTIMATOR = 100
LEARNING_RATE = 0.1
MAX_DEPTH = 3

# Data and ML utilities
import pandas as pd  
from IPython.display import display  
from sklearn.preprocessing import LabelEncoder  
from sklearn.model_selection import cross_validate, cross_val_score  
from sklearn.decomposition import PCA  
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA  
from sklearn.ensemble import GradientBoostingClassifier  
from sklearn.pipeline import Pipeline  


import sys
from pathlib import Path

PROJECT_ROOT = "../../"
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
from LdaBoost import LdaBoost

# Load dataset
DATA_PATH = str(Path(PROJECT_ROOT) / "real_datasets" / "iris" / "Data" / "IRIS.csv")
data = pd.read_csv(DATA_PATH)

y = data["species"]
X = data.drop(columns="species").copy()

# Encode labels to integers
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

# Quick peek of the data
display(data.head())
print(f"Dataset shape: {data.shape}")


,sepal_length,sepal_width,petal_length,petal_width,species
0,5.1,3.5,1.4,0.2,Iris-setosa
1,4.9,3.0,1.4,0.2,Iris-setosa
2,4.7,3.2,1.3,0.2,Iris-setosa
3,4.6,3.1,1.5,0.2,Iris-setosa
4,5.0,3.6,1.4,0.2,Iris-setosa


Dataset shape: (150, 5)


## Dataset and preprocessing

- Source: Iris dataset (`IRIS.csv`)
- Features: `sepal_length`, `sepal_width`, `petal_length`, `petal_width`
- Target: `species` (encoded to integers)
- Preprocessing: label encoding only
- We also preserve the original train/test split definition for parity (not used in CV results).


In [12]:
# Preserve the original split strategy for parity (80/20, random_state=1)
from sklearn.model_selection import train_test_split  
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=1
)
# Ensure arrays for shape reporting
X_train = np.asarray(X_train)
X_test = np.asarray(X_test)
y_train = np.asarray(y_train)
y_test = np.asarray(y_test)
print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")


Train shape: (120, 4), Test shape: (30, 4)


## LDA+GBM vs PCA+GBM

We compare two pipelines:
- LDA followed by Gradient Boosting
- PCA followed by Gradient Boosting

We use 10-fold cross-validation with `accuracy`.


In [13]:
from dataclasses import dataclass, field
from typing import Any, Dict

@dataclass
class DuoBoostCV:
    """Compare LDA+GBM vs PCA+GBM using cross-validation.

    Parameters
    ----------
    seed : int
        Seed propagated to models where applicable.
    """
    seed: int = RANDOM_SEED
    _models: Dict[str, Any] = field(init=False, repr=False)

    def _build_models(self, n_features: int) -> Dict[str, Any]:
        half_components = max(1, n_features // 2)
        return {
            "PCA + GBM": Pipeline([
                ("pca", PCA(n_components=half_components, random_state=self.seed)),
                ("gb", GradientBoostingClassifier(
                    n_estimators=N_ESTIMATOR,
                    learning_rate=LEARNING_RATE,
                    max_depth=MAX_DEPTH,
                    random_state=self.seed,
                )),
            ]),
            "LDA + GBM": Pipeline([
                ("lda", LDA(n_components=None)),
                ("gb", GradientBoostingClassifier(
                    n_estimators=N_ESTIMATOR,
                    learning_rate=LEARNING_RATE,
                    max_depth=MAX_DEPTH,
                    random_state=self.seed,
                )),
            ]),
        }

    def fit(self, X_in: np.ndarray, y_in: np.ndarray, **cv_kwargs) -> pd.DataFrame:
        X_arr = np.asarray(X_in)
        y_arr = np.asarray(y_in)

        cv_kwargs = cv_kwargs.copy()
        cv_kwargs.setdefault("scoring", "accuracy")
        cv_kwargs.setdefault("cv", 10)
        cv_kwargs.setdefault("n_jobs", -1)
        cv_kwargs.setdefault("return_train_score", True)

        self._models = self._build_models(X_arr.shape[1])
        scores: Dict[str, Dict[str, float]] = {}

        for name, model in self._models.items():
            cv_res = cross_validate(model, X_arr, y_arr, **cv_kwargs)
            metric_key = next(k for k in cv_res if k.startswith("test_"))
            metric_name = metric_key.replace("test_", "")
            scores[name] = {
                f"mean_{metric_name}": float(cv_res[metric_key].mean()),
                f"std_{metric_name}": float(cv_res[metric_key].std()),
            }

        self.results_ = pd.DataFrame(scores).T.sort_values(by=f"mean_{metric_name}", ascending=False)
        return self.results_

    def __call__(self, X_in: np.ndarray, y_in: np.ndarray, **cv_kwargs) -> pd.DataFrame:
        return self.fit(X_in, y_in, **cv_kwargs)

trainer = DuoBoostCV(seed=RANDOM_SEED)
results_duo = trainer(X, y_encoded, cv=10, scoring="accuracy", n_jobs=-1, return_train_score=True)
display(results_duo.round(4))


,mean_score,std_score
LDA + GBM,0.9733,0.0533
PCA + GBM,0.9333,0.0596


## Baseline Gradient Boosting

We evaluate a standalone Gradient Boosting classifier with the standardized constants.


In [16]:
gb_clf = GradientBoostingClassifier(
    n_estimators=N_ESTIMATOR,
    learning_rate=LEARNING_RATE,
    max_depth=MAX_DEPTH,
    random_state=RANDOM_SEED,
)

cv_scores = cross_validate(
    gb_clf,
    X,
    y_encoded,
    cv=10,
    scoring="accuracy",
    n_jobs=-1,
    return_train_score=True,
)

test_key = next(k for k in cv_scores if k.startswith("test_"))
metric_name = test_key.replace("test_", "")
mean_acc = float(cv_scores[test_key].mean())
std_acc = float(cv_scores[test_key].std())
print(f"GBM {metric_name}: {mean_acc:.4f} ± {std_acc:.4f}")


GBM score: 0.9600 ± 0.0442


## LdaBoost

We evaluate the custom `LdaBoost` classifier imported from the local package.


In [ ]:
ldaboost = LdaBoost(
    n_estimators=N_ESTIMATOR,
    learning_rate=LEARNING_RATE,
    max_depth=MAX_DEPTH,
    random_state=RANDOM_SEED,
)

accs = ldaboost.cross_validate(X=np.asarray(X), y=np.asarray(y_encoded), cv=10)
print(f"LdaBoost accuracy: {np.mean(accs):.4f} ± {np.std(accs):.4f}")


LdaBoost accuracy: 0.9667 ± 0.0537
